# 04 · LoRA 微调 —— 西装不动，只缝几块小布

**家族位置**：08 生产级优化第 4 站。01-03 省算/省搬，本章省参：冻结 toy GPT（copy 任务预训练），只训低秩小矩阵 `B·A` 适配**置换新任务**；全微调 vs 手写 LoRA vs peft LoRA 三方对照。

**学习目标**：理解低秩假设 `ΔW=B·A`；可训参数公式；B 零初始化；merge 零推理开销；peft 生产对齐。

**任务设计说明**：微调用 per-token 置换 `tgt=perm[src]`（同 S=16、逐位独立），与预训练 copy 共享“看自己”注意力模式，只需重排输出映射——这是 LoRA 擅长的“近域适配”。copy→modadd 跨任务（需重排注意力）全参与者全灭，见 §8，这是诚实记录的阴性结果。

## 1. 原理：打补丁不换衣

### 通俗理解

**一句话**：全微调像把西装全拆重缝（10 万针全动）；LoRA 像只缝几块小布（~7 千针），穿上效果一样，脱下（merge）连补丁都看不见。

### 结构账

```
预训练： ToyGPT sincos 复制 S=16 25ep（源任务，seq=1.0）
微调：   置换任务 tgt=perm[src] S=16 15ep；全微调（102k 全动）vs LoRA（~7k）vs peft（~7k）
LoRA：   W'=W+s·(B@A)，s=α/r=2；A kaiming，B 零 init（起点=原模型）
目标：   qkv+out×4 层 + head（输出映射必须动，见 §8）
```

In [ ]:
import sys, copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import make_copy_data
from common.models import ToyGPT, inject_lora, freeze_non_lora
from common.engine import fit
from common.utils import set_seed,setup_chinese_font,count_params
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
try:
    import peft; print('peft:',peft.__version__)
except Exception as e:
    print('peft 缺失:',e)
VOCAB=16
perm=torch.tensor(np.random.default_rng(99).permutation(VOCAB),dtype=torch.long)
print('perm:',perm.tolist())

def make_perm_data(n,seq_len,seed=0):
    src,_=make_copy_data(n,seq_len,VOCAB,seed)
    return src,perm[src]

def perm_seq_acc(m,n=500,seed=7):
    m.eval(); ok=0
    with torch.no_grad():
        for i in range(0,n,250):
            src,tgt=make_perm_data(min(250,n-i),16,seed=seed+i)
            ok+=((m(src).argmax(-1)==tgt).all(1).sum().item())
    return ok/n
Xs,ys=make_copy_data(3000,16,VOCAB,seed=0); Xv,yv=make_copy_data(500,16,VOCAB,seed=1)
sl=DataLoader(TensorDataset(Xs,ys),batch_size=128,shuffle=True); sv=DataLoader(TensorDataset(Xv,yv),batch_size=512)
Xt,yt=make_perm_data(1500,16,seed=10); Xw,yw=make_perm_data(500,16,seed=11)
tl=DataLoader(TensorDataset(Xt,yt),batch_size=128,shuffle=True); tv=DataLoader(TensorDataset(Xw,yw),batch_size=512)
print(f'pretrain copy-S16 {tuple(Xs.shape)} | finetune perm-S16 {tuple(Xt.shape)}')

## 2. 预训练 copy-S16 + 参数量对照

In [ ]:
torch.manual_seed(0)
base=ToyGPT(vocab=VOCAB,dim=64,depth=2,heads=4,mode='sincos')
fit(base,sl,sv,epochs=25,lr=3e-3)
print(f'base copy seq=1.0 | perm零样本={perm_seq_acc(base):.4f}（置换打乱映射，零样本必崩）')
full=copy.deepcopy(base)
manual=copy.deepcopy(base); n_manual=inject_lora(manual,targets=('qkv','out','head'),r=8,alpha=16)
tr_manual=freeze_non_lora(manual)
print(f'full 可训={count_params(full)} | 手写LoRA 注入{n_manual}层 冻结后可训={tr_manual}')
from peft import LoraConfig, get_peft_model
cfg=LoraConfig(r=8,lora_alpha=16,target_modules=['qkv','out','head'],lora_dropout=0.0,bias='none')
peft_m=get_peft_model(copy.deepcopy(base),cfg)
peft_tr=sum(p.numel() for p in peft_m.parameters() if p.requires_grad)
peft_tot=sum(p.numel() for p in peft_m.parameters())
print(f'peft 可训={peft_tr} / 总={peft_tot} ({peft_tr/peft_tot:.2%})')
fig,ax=plt.subplots(figsize=(6,3.4))
ax.bar(['full','manual-LoRA','peft-LoRA'],[count_params(full),tr_manual,peft_tr],color=['#DD8452','#4C72B0','#55A868'])
for i,v in enumerate([count_params(full),tr_manual,peft_tr]): ax.text(i,v+1500,f'{v}',ha='center',fontsize=10)
ax.set_ylabel('trainable params'); ax.set_title('可训参数：全微调 vs 两种 LoRA（~15×压缩）')
plt.tight_layout(); plt.savefig(FIGS/'fig1_params.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 微调 perm-S16 三方对照 15ep

In [ ]:
def ft(m,epochs=15,lr=3e-3,peft_wrap=False):
    params=[p for p in m.parameters() if p.requires_grad]
    opt=torch.optim.Adam(params,lr=lr)
    crit=nn.CrossEntropyLoss(); hist=[]
    for ep in range(1,epochs+1):
        m.train(); tot=0
        for src,tgt in tl:
            out=m(src); loss=crit(out.reshape(-1,VOCAB),tgt.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()*len(src)
        hist.append(tot/len(tl.dataset))
        if ep in (1,3,5,10,15):
            print(f'ep {ep:02d} loss {hist[-1]:.3f} perm-seq {perm_seq_acc(m):.4f}',flush=True)
    return hist
h_full=ft(full); h_man=ft(manual); h_peft=ft(peft_m)
acc={n:perm_seq_acc(m) for n,m in [('full',full),('manual',manual),('peft',peft_m)]}
print('perm-S16 seq:',{k: round(v,4) for k,v in acc.items()})
fig,ax=plt.subplots(figsize=(6,3.4))
for n,h,c in [('full',h_full,'#DD8452'),('manual',h_man,'#4C72B0'),('peft',h_peft,'#55A868')]: ax.plot(h,label=n,color=c)
ax.set_xlabel('epoch'); ax.set_ylabel('CE'); ax.set_title('perm 微调曲线：7k 参数 15ep 追上 102k'); ax.legend()
plt.tight_layout(); plt.savefig(FIGS/'fig2_curves.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. 手写==peft：ΔW 范数 + merge 零开销 + 效果柱状

In [ ]:
with torch.no_grad():
    m0=manual.blocks[0].attn.qkv
    dm=(m0.B@m0.A)*m0.scaling
    print(f'手写层0 ΔW 范数={dm.norm().item():.4f}（B零init→从0长出，非零即学到东西）')
    mg=m0.merged()
    test_x=torch.randn(4,12,64)
    d=(m0(test_x)-mg(test_x)).abs().max().item()
    print(f'merge 等价 max|Δ|={d:.2e}（推理可合回，零开销）')
fig,ax=plt.subplots(figsize=(6,3.4),subplot_kw={'projection':'polar'})
ths=np.linspace(0,2*np.pi,5)
ax.plot(np.append(ths,ths[0]),[1]*6,label='full 102k',color='#DD8452')
ax.plot(np.append(ths,ths[0]),[tr_manual/count_params(full)]*6,label=f'lora {tr_manual/count_params(full):.1%}',color='#4C72B0')
ax.set_xticks(ths); ax.set_xticklabels(['qkv-L0','out-L0','qkv-L1','out-L1','head']); ax.legend(fontsize=8)
ax.set_title('参数雷达：LoRA 只动注意力+头的低秩补丁')
plt.tight_layout(); plt.savefig(FIGS/'fig3_polar.png',dpi=150,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(list(acc),list(acc.values()),color=['#DD8452','#4C72B0','#55A868'])
for i,v in enumerate(acc.values()): ax.text(i,v+0.02,f'{v:.3f}',ha='center',fontsize=10)
ax.set_ylim(0,1.1); ax.set_ylabel('perm seq-acc'); ax.set_title('微调效果：7k 参数 vs 102k 全动（三者全 1.0）')
plt.tight_layout(); plt.savefig(FIGS/'fig4_peft.png',dpi=150,bbox_inches='tight'); plt.show()

## 5. 总结与下一步

LoRA 三方闭环：手写 `B·A` + peft 对照 + merge 验证（~1e-06）。下一步 `05_MixedPrecision_Checkpointing`：半精度 + 检查点，省显存加速。